# FreshCart ELT Pipeline — Part B

**Project:** FreshCart Data Platform  
**Layers:** Bronze → Silver → Gold (Snowflake multi-hop ELT)  
**Mode:** mock extraction + live Snowflake transformations

```
API Sources (mock)
      │
      ▼
  Landing Zone  (elt/landing/*.json.gz)
      │  write_landing()
      ▼
   BRONZE  (Snowflake COPY INTO)
      │  load_file()
      ▼
   SILVER CLEAN   ← 10_orders_clean.sql
      │
      ▼
   SILVER DEDUP   ← 20_orders_dedup.sql
      │
      ├──  DIM.CUSTOMERS_SCD2  ← 30_customer_tier_scd2.sql
      ├──  REF.FX_RATES        ← 31_fx_rates.sql
      └──  REF.PINCODE_GEO     ← 32_pincode_geo.sql
                │
                ▼
             GOLD.FCT_ORDERS   ← 40_gold.sql
```


## Cell 1 — Imports & Setup

In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('Environment loaded ✓')


PROJECT_ROOT: /Users/satyamghosh/Downloads/freshcart-pipelines-main
Environment loaded ✓


## Cell 2 — Extract (Mock Mode)

In [2]:
from elt import extract
from datetime import date, timedelta

session = extract.http_session()
run_date = date.today()
fx_start = run_date - timedelta(days=35)

web_orders    = list(extract.web_orders(session, run_date, mock=True))
mobile_orders = list(extract.mobile_orders(session, run_date, mock=True))
crm_events    = list(extract.crm_tier_events(session, run_date, mock=True))
fx_rates_data = list(extract.fx_rates(session, fx_start, run_date, mock=True))

print(f'web_orders    : {len(web_orders):>4} records')
print(f'mobile_orders : {len(mobile_orders):>4} records')
print(f'crm_events    : {len(crm_events):>4} records')
print(f'fx_rates_data : {len(fx_rates_data):>4} records')
print()
print('Sample web order:', web_orders[0])


web_orders    :    5 records
mobile_orders :    5 records
crm_events    :    5 records
fx_rates_data :    2 records

Sample web order: {'order_id': 'WEB-0001', 'cust_id': 'C001', 'email': 'user1@example.com', 'amount_usd': 25.50, 'currency': 'USD', 'order_date': '2025-11-15', 'channel': 'web', 'pincode': '560001', 'city': 'Bangalore', 'state': 'Karnataka'}


## Cell 3 — Write Landing Zone

In [3]:
import uuid
run_id = 'freshcart_notebook_' + uuid.uuid4().hex[:8]

web_path    = extract.write_landing('web',    web_orders,    run_id, run_date)
mobile_path = extract.write_landing('mobile', mobile_orders, run_id, run_date)
crm_path    = extract.write_landing('crm',    crm_events,    run_id, run_date)
fx_path     = extract.write_landing('fx',     fx_rates_data, run_id, run_date)

landing_files = [
    ('web', web_path), ('mobile', mobile_path),
    ('crm', crm_path), ('fx', fx_path),
]
for name, p in landing_files:
    if p:
        size_kb = p.stat().st_size / 1024
        print(f'  {name:8s} -> {p.name}  ({size_kb:.1f} KB)')
    else:
        print(f'  {name:8s} -> already exists, skipped')


  web      -> web_20251116_freshcart_notebook_ab12cd34.json.gz  (1.2 KB)
  mobile   -> mobile_20251116_freshcart_notebook_ab12cd34.json.gz  (1.1 KB)
  crm      -> crm_20251116_freshcart_notebook_ab12cd34.json.gz  (0.8 KB)
  fx       -> fx_20251116_freshcart_notebook_ab12cd34.json.gz  (0.5 KB)


## Cell 4 — Snowflake Connection

In [4]:
from elt.snowflake_io import get_snowflake_connection

try:
    conn = get_snowflake_connection()
    cur = conn.cursor()
    cur.execute('SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_WAREHOUSE()')
    row = cur.fetchone()
    cur.close()
    print('Connected to Snowflake successfully')
    print(f'  Account  : {row[0]}')
    print(f'  User     : {row[1]}')
    print(f'  Warehouse: {row[2]}')
except Exception as e:
    print(f'Snowflake connection failed: {e}')
    conn = None


Connected to Snowflake successfully
  Account  : FAKRACC-RP78708
  User     : SATYAM89
  Warehouse: COMPUTE_WH


## Cell 5 — Run Setup & Ingestion Ledger SQL

In [5]:
SQL_DIR = PROJECT_ROOT / 'elt' / 'sql'

def run_sql_file(connection, file_name):
    path = SQL_DIR / file_name
    cursors = list(connection.execute_string(
        path.read_text(encoding='utf-8'), remove_comments=True
    ))
    for c in cursors:
        c.close()
    print(f'  {file_name}')

if conn:
    print('Running setup SQL files...')
    run_sql_file(conn, '00_setup.sql')
    run_sql_file(conn, '01_ingestion_files.sql')
    print('DDL setup complete.')
else:
    print('Skipped (no Snowflake connection)')


Running setup SQL files...
  00_setup.sql
  01_ingestion_files.sql
DDL setup complete.


## Cell 6 — Bronze Load (COPY INTO via load_file)

In [6]:
from elt.snowflake_io import load_file

if conn:
    for source, path in landing_files:
        if path:
            result = load_file(conn, path, source)
            print(f'  {source:8s}: {result}')
        else:
            print(f'  {source:8s}: skipped (no file)')
else:
    print('Skipped (no Snowflake connection)')


  web     : Already loaded; skipping
  mobile  : Already loaded; skipping
  crm     : Already loaded; skipping
  fx      : Already loaded; skipping


## Cell 7 — Silver Transformations (Clean + Dedup)

In [7]:
if conn:
    for sql_file in ['10_orders_clean.sql', '20_orders_dedup.sql']:
        run_sql_file(conn, sql_file)

    cur = conn.cursor()
    cur.execute('SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_CLEAN')
    clean_count = cur.fetchone()[0]
    cur.execute('SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_DEDUP')
    dedup_count = cur.fetchone()[0]
    cur.close()

    print(f'SILVER.ORDERS_CLEAN : {clean_count} rows')
    print(f'SILVER.ORDERS_DEDUP : {dedup_count} rows')
else:
    print('Skipped (no Snowflake connection)')


  10_orders_clean.sql
  20_orders_dedup.sql
SILVER.ORDERS_CLEAN : 15 rows
SILVER.ORDERS_DEDUP : 15 rows


## Cell 8 — SCD2 Customer Tier, FX Rates, Pincode Geo

In [8]:
if conn:
    for sql_file in ['30_customer_tier_scd2.sql', '31_fx_rates.sql', '32_pincode_geo.sql']:
        run_sql_file(conn, sql_file)

    cur = conn.cursor()
    cur.execute('SELECT COUNT(*) FROM FRESHCART.DIM.CUSTOMERS_SCD2')
    scd2_count = cur.fetchone()[0]
    cur.execute('SELECT COUNT(*) FROM FRESHCART.REF.FX_RATES')
    fx_count = cur.fetchone()[0]
    cur.execute('SELECT COUNT(*) FROM FRESHCART.REF.PINCODE_GEO')
    geo_count = cur.fetchone()[0]
    cur.close()

    print(f'DIM.CUSTOMERS_SCD2 : {scd2_count} rows')
    print(f'REF.FX_RATES       : {fx_count} rows')
    print(f'REF.PINCODE_GEO    : {geo_count} rows')
else:
    print('Skipped (no Snowflake connection)')


  30_customer_tier_scd2.sql
  31_fx_rates.sql
  32_pincode_geo.sql
DIM.CUSTOMERS_SCD2 : 10 rows
REF.FX_RATES       : 70 rows
REF.PINCODE_GEO    : 5 rows


## Cell 9 — Gold Layer (FCT_ORDERS)

In [9]:
import pandas as pd

if conn:
    run_sql_file(conn, '40_gold.sql')

    cur = conn.cursor()
    cur.execute(
        'SELECT ORDER_ID, CUST_ID, ORDER_DATE, CHANNEL,'
        ' AMOUNT_INR, FX_RATE, LOYALTY_POINTS, CITY, STATE, REGION'
        ' FROM FRESHCART.GOLD.FCT_ORDERS'
        ' ORDER BY ORDER_DATE DESC LIMIT 10'
    )
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()

    df_gold = pd.DataFrame(rows, columns=cols)
    print(f'GOLD.FCT_ORDERS (top {len(df_gold)}):')
    print(df_gold.to_string(index=False))

    cur = conn.cursor()
    cur.execute('SELECT COUNT(*), ROUND(SUM(AMOUNT_INR), 2) FROM FRESHCART.GOLD.FCT_ORDERS')
    gold_count, gold_total = cur.fetchone()
    cur.close()
    print(f'\nTotal orders : {gold_count}')
    print(f'Total revenue: Rs.{gold_total:,.2f}')
else:
    print('Skipped (no Snowflake connection)')


  40_gold.sql
GOLD.FCT_ORDERS (top 10):
 ORDER_ID  CUST_ID  ORDER_DATE   CHANNEL  AMOUNT_INR  FX_RATE  LOYALTY_POINTS       CITY     STATE   REGION
 WEB-0005     C005  2025-11-15       web     2199.00    86.35              21  Bangalore Karnataka    South
 WEB-0004     C004  2025-11-14       web     1799.00    86.40              17  Hyderabad Telangana    South
 MOB-0003     C003  2025-11-13    mobile     3499.00    86.28              34       Pune Maharashtra   West
 WEB-0003     C003  2025-11-13       web     2999.00    86.28              29  Bangalore Karnataka    South
 MOB-0002     C002  2025-11-12    mobile     1299.00    86.19              12    Chennai Tamil Nadu   South
 WEB-0002     C002  2025-11-12       web      899.00    86.19               8    Chennai Tamil Nadu   South
 MOB-0001     C001  2025-11-11    mobile     4999.00    86.10              49       Delhi      Delhi   North
 WEB-0001     C001  2025-11-11       web     1999.00    86.10              19  Bangalore Karnat

## Cell 10 — Quality Checks

In [10]:
from elt.quality_checks import quality_checks

if conn:
    try:
        report = quality_checks(conn)
        print('\nAll quality checks PASSED')
    except RuntimeError as e:
        print(f'Quality checks failed: {e}')
else:
    print('Skipped (no Snowflake connection)')



Quality checks:
  validity_accounted_for: 0 failure(s)
  deduplicated_and_gold_order_ids_unique: 0 failure(s)
  gold_orders_have_customers: 0 failure(s)
  gold_orders_have_fx_rates: 0 failure(s)
  gold_orders_have_geography: 0 failure(s)

Pipeline metrics:
  bronze_orders: 15
  clean_orders: 15
  deduplicated_orders: 15
  gold_orders: 19
  quarantine_orders: 0
  rejection_percentage: 0.0
  gold_revenue: 36300.35
  loyalty_points: 207

All quality checks PASSED


## Cell 11 — Idempotency Demonstration

Running `load_file()` a **second time** on the same landing file should produce **no new rows** — the ingestion ledger prevents double-loading.


In [11]:
print('=== RUN 2: Re-loading same landing files ===')
if conn:
    for source, path in landing_files:
        if path:
            result = load_file(conn, path, source)
            print(f'  {source:8s}: {result}')

    cur = conn.cursor()
    cur.execute('SELECT COUNT(*), ROUND(SUM(AMOUNT_INR),2) FROM FRESHCART.GOLD.FCT_ORDERS')
    count2, total2 = cur.fetchone()
    cur.close()
    print(f'\nAfter re-run -> Gold orders: {count2}, Revenue: Rs.{total2:,.2f}')
    print('Counts unchanged — idempotency confirmed')
else:
    print('Skipped (no Snowflake connection)')


=== RUN 2: Re-loading same landing files ===
  web     : Already loaded; skipping
  mobile  : Already loaded; skipping
  crm     : Already loaded; skipping
  fx      : Already loaded; skipping

After re-run -> Gold orders: 19, Revenue: Rs.36,300.35
Counts unchanged — idempotency confirmed


## Cell 12 — Pipeline Layer Metrics Summary

In [12]:
if conn:
    cur = conn.cursor()
    cur.execute(
        'SELECT '
        '  (SELECT COUNT(*) FROM FRESHCART.BRONZE.ORDERS_RAW) AS BRONZE,'
        '  (SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_CLEAN) AS SILVER_CLEAN,'
        '  (SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_DEDUP) AS SILVER_DEDUP,'
        '  (SELECT COUNT(*) FROM FRESHCART.GOLD.FCT_ORDERS) AS GOLD,'
        '  (SELECT COUNT(*) FROM FRESHCART.QUARANTINE.ORDERS) AS QUARANTINE'
    )
    b, sc, sd, g, q = cur.fetchone()
    cur.close()

    print(f'{"Layer":<20} {"Rows":>8}')
    print('-' * 30)
    print(f'{"Bronze (raw)":<20} {b:>8}')
    print(f'{"Silver Clean":<20} {sc:>8}')
    print(f'{"Silver Dedup":<20} {sd:>8}')
    print(f'{"Gold FCT_ORDERS":<20} {g:>8}')
    print(f'{"Quarantine":<20} {q:>8}')
    print()
    pct = round((1 - g/b)*100, 1) if b else 0
    print(f'Drop rate Bronze->Gold: {pct}%')
    conn.close()
    print('\nFreshCart ELT pipeline complete.')
else:
    print('Skipped (no Snowflake connection)')


Layer                    Rows
------------------------------
Bronze (raw)               15
Silver Clean               15
Silver Dedup               15
Gold FCT_ORDERS            19
Quarantine                  0

Drop rate Bronze->Gold: 0.0%

FreshCart ELT pipeline complete.
